# Phase 6 (post hoc): a larger classifier in P18's pipeline (phase6_bigger_classifier, P22)

PREREGISTRATION §10 v21, P22. Phase 2's ResNet1D (6.7 times the cnn's parameters; it lost to the cnn at phase 2) is
trained on P13's data with phase 2's learning rate for it and the shared budget, for no augmentation, the cVAE and the
WGAN-GP, seeds 0-9. Huang's stage and P16's veto are chosen for each ResNet ensemble on P14's patients by P20's rule; P19's
patients and one DS2 read compare it with P18's cnn ensemble for the same condition.

| `PART` | accelerator | inputs | what it does |
|---|---|---|---|
| `run` | GPU | `ecgaug-repo`, `ecgaug-v5-external`, `ecgaug-banks`, `ecgaug-p12-banks`, `ecgaug-p13-runs`, `ecgaug-p15-runs`, `ecgaug-p18-runs`; output of `phase6-rule-tuning-build` | the 30 ResNet runs, the setting chosen on P14's patients, scores on P19's |
| `test` | CPU | `ecgaug-repo`, `ecgaug-v5-external`, `ecgaug-p13-runs`, `ecgaug-p15-runs`, `ecgaug-p18-runs`; output of the `run` session | the one posthoc DS2 read (logged) |

`test` needs `members.json` and `choice.json` from `run` in the repository. Download the zip in Output and run
`python kaggle/ingest_runs.py <zip>`.


In [ ]:
# --- Parameters: edit only this cell ---------------------------------------------------------
PART = "run"                          # "run" (GPU) or "test" (CPU, the DS2 read)
RUN_TESTS = False                     # the full test suite first; nothing else runs if it fails
INPUTS_DIR = "/kaggle/input"          # attached datasets: banks, checkpoints (runs/), external data (data/)
GIT_REF = "main"                      # branch name or full commit SHA to run
DEVICE = "auto"                       # "auto" uses the GPU when the session has one
ENV_MODE = "locked"                   # "locked" = exact uv.lock versions; "kaggle" = preinstalled packages (faster, debugging only)
RAW_DATASET_DIR = "/kaggle/input"     # attached mitdb-raw files; if none, downloaded from PhysioNet's AWS mirror (checksums checked)
REPO_URL = "https://github.com/akashsaha477/Gen_Ai_Project.git"


In [ ]:
import os, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None                      # public repository, or the secret is not attached
if token:
    askpass = WORK / ".git-askpass.sh"
    askpass.write_text('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
    askpass.chmod(0o700)
    env.update(GIT_ASKPASS=str(askpass), GITHUB_TOKEN=token)   # the token is never printed or put in a URL
    url = REPO_URL.replace("https://", "https://x-access-token@")

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True, env=env)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO, check=True, env=env)
ref = GIT_REF if len(GIT_REF) == 40 else f"origin/{GIT_REF}"
subprocess.run(["git", "checkout", "--quiet", "--detach", ref], cwd=REPO, check=True)
sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
print("running commit", sha)

In [ ]:
configs = ["configs/phase6_bigger_classifier/bigger_classifier.yaml"]
extra = ["--part", PART, "--device", DEVICE]
if PART == "run":
    extra += ["--beats-dir", INPUTS_DIR]   # P20's built beats, from the attached phase6-rule-tuning-build output
# external data only from the ecgaug-v5-external dataset: the build output's repo/ folder holds a large data/ folder too
external = sorted(str(p) for p in Path(INPUTS_DIR).rglob("ecgaug-v5-external") if p.is_dir())
if not external:
    raise SystemExit("attach the ecgaug-v5-external dataset")
cmd = [sys.executable, "kaggle/kaggle_run.py", "--cli", "ecgaug.experiments.phase6_bigger_classifier", "--env", ENV_MODE, "--out", str(WORK),
       "--config", *configs, "--raw-dir", RAW_DATASET_DIR, "--artifacts-dir", INPUTS_DIR, "--data-dir", external[0]]
if RUN_TESTS:
    cmd += ["--tests"]
cmd += ["--", *extra]
if PART == "test":
    env["ALLOW_TEST_POSTHOC"] = "1"   # the one posthoc DS2 read of PREREGISTRATION v21 (P22); it is logged in docs/TEST_ACCESS_LOG.md
code = subprocess.run(cmd, cwd=REPO, env=env).returncode
print("finished with exit code", code, "(2 = the run was refused, read the message above)")